# Seed check — is ConvNeXt-Tiny really better than Xception?

In `04_model_comparison`, ConvNeXt-Tiny and Xception **tied** on our main rule (10 silent missed tumors each),
and the macro-F1 difference was only 0.2% (about 3 test images). That can be random luck from one training run.

So we train **only these two models again with 2 new random seeds** (1 and 2).
Together with the first run (seed 42) we get **3 runs per model** and compare the **average**.

**What changes between seeds:** the random start of the new last layer, the order of training images, and the augmentation.
**What stays the same:** the data, the train/validation split, the test set, and all settings.

**Kaggle:** same settings as before (dataset, GPU T4, Internet on) → **Save Version → Save & Run All** (about 40 minutes).

### Decision rule (written before seeing results)
1. Lowest **average silent missed tumors** over 3 seeds.
2. If the averages are within 2 scans of each other: highest **average macro-F1**.
3. For the app we use the run of the winning model with the **best validation accuracy**
   (never chosen by test results — that would be cheating).

In [ ]:
!pip install -q grad-cam timm

In [ ]:
import os, random, time, copy, json
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## Settings
The same for every model. Only `MODELS` changes.

In [ ]:
MODELS = ["convnext_tiny", "legacy_xception"]
SEED, IMG_SIZE, BATCH_SIZE, EPOCHS, LR, VAL_RATIO = 42, 224, 32, 12, 1e-4, 0.15
PRETRAINED = True
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
OUT_DIR = Path("/kaggle/working"); INPUT_ROOT = Path("/kaggle/input")
(OUT_DIR / "models").mkdir(parents=True, exist_ok=True)

# Safety rule — same numbers as core/predictor.py in the app
MIN_CONFIDENCE, MIN_MARGIN, NOTUMOR_TRUST = 0.80, 0.20, 0.95
CPU_THREADS = 2          # like a free Hugging Face Space
MAX_CPU_SECONDS = 1.0    # rule 1

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

## Step 1 — Load the data once (into memory)
We resize every image to 224×224 **once** and keep it in RAM. This makes training 6 models much faster,
because we don't read and resize the files again for every epoch.

In [ ]:
def find_data_dir(root):
    for p in sorted(root.rglob("Training")):
        if p.is_dir() and (p.parent / "Testing").is_dir():
            return p.parent
    raise FileNotFoundError("Training/ + Testing/ not found. Did you add the dataset?")

DATA_DIR = find_data_dir(INPUT_ROOT)
IMG_EXT = {".jpg", ".jpeg", ".png"}
def list_images(d):
    return pd.DataFrame([{"path": str(f), "label": c.name}
                         for c in sorted(p for p in d.iterdir() if p.is_dir())
                         for f in c.iterdir() if f.suffix.lower() in IMG_EXT])

train_full_df = list_images(DATA_DIR / "Training")
test_df = list_images(DATA_DIR / "Testing")
CLASS_NAMES = sorted(train_full_df.label.unique())
C2I = {c: i for i, c in enumerate(CLASS_NAMES)}
NT = C2I["notumor"]

train_df, val_df = train_test_split(train_full_df, test_size=VAL_RATIO,
                                    stratify=train_full_df.label, random_state=SEED)

def load_all(df):
    return [Image.open(p).convert("RGB").resize((IMG_SIZE, IMG_SIZE)) for p in df.path]

t0 = time.time()
train_imgs, val_imgs, test_imgs = load_all(train_df), load_all(val_df), load_all(test_df)
print(f"train {len(train_imgs)} | val {len(val_imgs)} | test {len(test_imgs)} | loaded in {time.time()-t0:.0f}s")

## Step 2 — Same preprocessing for every model

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class MemDataset(Dataset):
    def __init__(self, imgs, labels, tf):
        self.imgs, self.labels, self.tf = imgs, [C2I[l] for l in labels], tf
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return self.tf(self.imgs[i]), self.labels[i]

def loader(imgs, df, tf, shuffle, seed=SEED):
    g = torch.Generator(); g.manual_seed(seed)
    return DataLoader(MemDataset(imgs, df.label, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=2, generator=g)

val_loader = loader(val_imgs, val_df, eval_tf, False)
test_loader = loader(test_imgs, test_df, eval_tf, False)
test_y = torch.tensor([C2I[l] for l in test_df.label])

## Step 3 — Helper functions
- `train_model`: same training loop as before (AdamW, cosine schedule, mixed precision, keep best validation epoch)
- `fit_temperature`: temperature scaling (honest confidence)
- `evaluate`: all the numbers we compare
- `cpu_speed`: time for one prediction + heatmap on 2 CPU cores (like free hosting)

In [ ]:
def cam_layer_name(model):
    # the last feature stage of the network; works for all timm CNNs we use
    return model.feature_info[-1]["module"]

def train_model(name):
    set_seed(SEED)
    model = timm.create_model(name, pretrained=PRETRAINED, num_classes=len(CLASS_NAMES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    tr_loader = loader(train_imgs, train_df, train_tf, True)
    best_acc, best_state, hist = 0, None, []
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train()
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", enabled=use_amp):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v_logits, v_y = get_logits(model, val_loader)
        acc = (v_logits.argmax(1) == v_y).float().mean().item()
        hist.append(acc)
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
        print(f"  {name} epoch {ep:2d}/{EPOCHS} val acc {acc:.4f}{'  <- best' if acc == best_acc else ''}")
    model.load_state_dict(best_state)
    return model, best_acc, hist, time.time() - t0

@torch.no_grad()
def get_logits(model, dl):
    model.eval(); L, Y = [], []
    for x, y in dl:
        L.append(model(x.to(device)).float().cpu()); Y.append(y)
    return torch.cat(L), torch.cat(Y)

def fit_temperature(logits, y):
    log_T = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.1, max_iter=200)
    def closure():
        opt.zero_grad(); l = F.cross_entropy(logits / log_T.exp(), y); l.backward(); return l
    opt.step(closure)
    return log_T.exp().item()

def ece(probs, y, n_bins=15):
    conf, pred = probs.max(1); correct = pred.eq(y).float(); e = 0.0
    for lo, hi in zip(torch.linspace(0, 1, n_bins + 1)[:-1], torch.linspace(0, 1, n_bins + 1)[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.float().mean() * (conf[m].mean() - correct[m].mean()).abs()
    return float(e)

def evaluate(logits, T):
    probs = F.softmax(logits / T, 1)
    conf, pred = probs.max(1)
    top2 = probs.topk(2, 1).values
    refer = (conf < MIN_CONFIDENCE) | ((top2[:, 0] - top2[:, 1]) < MIN_MARGIN) | ((pred == NT) & (probs[:, NT] < NOTUMOR_TRUST))
    wrong = pred != test_y
    is_tumor = test_y != NT
    missed = is_tumor & (pred == NT)                  # tumor called "no tumor"
    false_alarm = (~is_tumor) & (pred != NT)          # normal called "tumor"
    recalls = recall_score(test_y, pred, average=None, labels=range(len(CLASS_NAMES)))
    return {
        "test_acc": accuracy_score(test_y, pred),
        "macro_f1": f1_score(test_y, pred, average="macro"),
        **{f"recall_{c}": r for c, r in zip(CLASS_NAMES, recalls)},
        "tumor_sensitivity": 1 - missed.sum().item() / is_tumor.sum().item(),
        "missed_tumors": missed.sum().item(),
        "silent_missed_tumors": (missed & ~refer).sum().item(),
        "false_alarms": false_alarm.sum().item(),
        "ece": ece(probs, test_y),
        "flagged_pct": refer.float().mean().item(),
        "errors_flagged_pct": (wrong & refer).sum().item() / max(wrong.sum().item(), 1),
        "acc_unflagged": (~wrong[~refer]).float().mean().item(),
    }, pred

def cpu_speed(model, layer, n=10):
    torch.set_num_threads(CPU_THREADS)
    m = copy.deepcopy(model).cpu().eval()
    cam = GradCAM(model=m, target_layers=[m.get_submodule(layer)])
    x = eval_tf(test_imgs[0]).unsqueeze(0)
    with torch.no_grad(): m(x)  # warm-up
    t = time.time()
    for _ in range(n):
        with torch.no_grad(): idx = int(m(x).argmax())
        cam(input_tensor=x, targets=[ClassifierOutputTarget(idx)])
    torch.set_num_threads(os.cpu_count())
    return (time.time() - t) / n

## Step 4 — Train with a seed parameter
Same training loop as before, but the seed is a parameter. The data split stays the same (seed 42).

In [ ]:
def train_model_seed(name, seed):
    set_seed(seed)
    model = timm.create_model(name, pretrained=PRETRAINED, num_classes=len(CLASS_NAMES)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    use_amp = device == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    tr_loader = loader(train_imgs, train_df, train_tf, True, seed=seed)
    best_acc, best_state = 0, None
    for ep in range(1, EPOCHS + 1):
        model.train()
        for x, y in tr_loader:
            x, y = x.to(device), y.to(device)
            with torch.autocast("cuda", enabled=use_amp):
                loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        v_logits, v_y = get_logits(model, val_loader)
        acc = (v_logits.argmax(1) == v_y).float().mean().item()
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
        print(f"  {name} seed {seed} epoch {ep:2d}/{EPOCHS} val acc {acc:.4f}")
    model.load_state_dict(best_state)
    return model, best_acc

EXTRA_SEEDS = [1, 2]
rows = []
for name in MODELS:
    for seed in EXTRA_SEEDS:
        print(f"\n===== {name} | seed {seed} =====")
        model, best_val = train_model_seed(name, seed)
        layer = cam_layer_name(model)
        v_logits, v_y = get_logits(model, val_loader)
        T = fit_temperature(v_logits, v_y)
        t_logits, _ = get_logits(model, test_loader)
        metrics, _ = evaluate(t_logits, T)
        path = OUT_DIR / "models" / f"{name}_seed{seed}.pth"
        torch.save({"model_name": name, "state_dict": model.state_dict(), "class_names": CLASS_NAMES,
                    "img_size": IMG_SIZE, "mean": MEAN, "std": STD, "temperature": T,
                    "cam_layer": layer, "test_accuracy": float(metrics["test_acc"]), "seed": seed}, path)
        rows.append({"model": name, "seed": seed, "best_val_acc": best_val, **metrics})
        print({k: round(v, 4) if isinstance(v, float) else v for k, v in rows[-1].items()})
        del model; torch.cuda.empty_cache()

## Step 5 — Add the first run (seed 42) and compare averages
The seed-42 numbers are copied from the output of `04_model_comparison`.

In [ ]:
SEED42 = [
    {"model": "convnext_tiny",   "seed": 42, "best_val_acc": 0.9940, "test_acc": 0.959375, "macro_f1": 0.958779,
     "recall_glioma": 0.8475, "tumor_sensitivity": 0.9900, "missed_tumors": 12, "silent_missed_tumors": 10,
     "flagged_pct": 0.0150, "errors_flagged_pct": 0.1846, "ece": 0.0332},
    {"model": "legacy_xception", "seed": 42, "best_val_acc": 0.9940, "test_acc": 0.957500, "macro_f1": 0.956660,
     "recall_glioma": 0.8425, "tumor_sensitivity": 0.9817, "missed_tumors": 22, "silent_missed_tumors": 10,
     "flagged_pct": 0.0237, "errors_flagged_pct": 0.3824, "ece": 0.0317},
]
cols = ["model", "seed", "best_val_acc", "test_acc", "macro_f1", "recall_glioma", "tumor_sensitivity",
        "missed_tumors", "silent_missed_tumors", "flagged_pct", "errors_flagged_pct", "ece"]
allruns = pd.concat([pd.DataFrame(SEED42), pd.DataFrame(rows)], ignore_index=True)[cols]
allruns.to_csv(OUT_DIR / "seed_runs.csv", index=False)
display(allruns.round(4))

summary = allruns.groupby("model")[cols[3:]].agg(["mean", "std"])
summary.to_csv(OUT_DIR / "seed_summary.csv")
display(summary.round(4))

## Step 6 — Apply the decision rule

In [ ]:
m = allruns.groupby("model").agg(silent=("silent_missed_tumors", "mean"), f1=("macro_f1", "mean"),
                                 f1_std=("macro_f1", "std"), missed=("missed_tumors", "mean"))
display(m.round(4))
a, b = m.sort_values("silent").index[:2]
if m.loc[b, "silent"] - m.loc[a, "silent"] > 2:
    WINNER = a
    print(f"Rule 1: {a} has clearly fewer silent missed tumors ({m.loc[a,'silent']:.1f} vs {m.loc[b,'silent']:.1f}).")
else:
    WINNER = m["f1"].idxmax()
    print(f"Rule 1 is a tie (within 2 scans) -> Rule 2: higher average macro-F1 -> {WINNER}")
diff = abs(m.loc["convnext_tiny", "f1"] - m.loc["legacy_xception", "f1"])
noise = max(m["f1_std"])
print(f"Macro-F1 difference {diff:.4f} vs seed-to-seed std {noise:.4f} ->",
      "difference is larger than the noise" if diff > noise else "difference is within the noise (both models are equally good)")

best_run = allruns[allruns.model == WINNER].sort_values("best_val_acc", ascending=False).iloc[0]
print(f"\nWinner: {WINNER}. Use the run with the best VALIDATION accuracy: seed {int(best_run.seed)}")

## Step 7 — Save the final model for the app
If the best run is seed 42, use `best_model.pth` from the `04_model_comparison` output (it is that model).
Otherwise this cell copies the right file to `best_model.pth` here.
Then shrink it on your laptop with `python scripts/shrink_model.py` (makes it small enough for GitHub).

In [ ]:
import shutil
s = int(best_run.seed)
if s == 42:
    print(f"Use best_model.pth from 04_model_comparison ({WINNER}, seed 42). Nothing to copy here.")
else:
    shutil.copy(OUT_DIR / "models" / f"{WINNER}_seed{s}.pth", OUT_DIR / "best_model.pth")
    print(f"Saved /kaggle/working/best_model.pth = {WINNER} seed {s}")